# Exercitiul 1 - Clasificare Binara (FARA fisier adnotari, imagini secventiale)

**Varianta:** Imaginile sunt deja decupate (fete), organizate **consecutiv** pe personaj.
Exemplu:
- 0001-0055: DeeDee
- 0056-0110: Louie
- 0111-0165: Dexter
- 0166-0220: Tommy

Nu exista fisier cu adnotari - etichetele se deduc din intervalul de fisiere.

**Scor:** acuratete × nr_puncte_maxim

In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from copy import deepcopy
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CONFIGURARE - MODIFICA DOAR ACEASTA SECTIUNE
# ============================================================

EX1_DIR = "data/ex1"   # directorul cu imaginile

# Defineste intervalele de fisiere pentru fiecare personaj
# Format: (index_start, index_end_inclusiv, nume_personaj, eticheta_clasa)
# Eticheta: 1 = clasa pozitiva, 0 = clasa negativa
PERSONAJE = [
    (1,   55,  "DeeDee", 0),   # <-- modifica dupa enunt
    (56,  110, "Louie",  1),
    (111, 165, "Dexter", 0),
    (166, 220, "Tommy",  1),
]

# Primele N imagini per personaj -> train, restul -> test
NR_TRAIN_PER_PERSONAJ = 40   # <-- modifica dupa enunt (ex. 40 train, 15 test)

IMG_EXT             = ".jpg"          # sau ".png"
IMAGE_SIZE          = (128, 128)
HOG_ORIENTATIONS    = 9
HOG_PIX_PER_CELL    = (8, 8)
HOG_CELLS_PER_BLOCK = (3, 3)

In [ ]:
# ============================================================
# CONSTRUIRE LISTA IMAGINI CU ETICHETE
# ============================================================

all_entries = []  # (filepath, label, personaj)

for (start, end, personaj, label) in PERSONAJE:
    for idx in range(start, end + 1):
        fname = f"{idx:04d}{IMG_EXT}"
        fpath = os.path.join(EX1_DIR, fname)
        all_entries.append((fpath, label, personaj))

print(f"Total intrari: {len(all_entries)}")
print("Primele 3:", all_entries[:3])

In [ ]:
# ============================================================
# IMPARTIRE TRAIN / TEST per personaj
# ============================================================

train_entries = []
test_entries  = []

for (start, end, personaj, label) in PERSONAJE:
    entries_pers = [(os.path.join(EX1_DIR, f"{i:04d}{IMG_EXT}"), label, personaj)
                    for i in range(start, end + 1)]
    train_entries.extend(entries_pers[:NR_TRAIN_PER_PERSONAJ])
    test_entries.extend(entries_pers[NR_TRAIN_PER_PERSONAJ:])

print(f"Train: {len(train_entries)} | Test: {len(test_entries)}")

In [ ]:
# ============================================================
# INCARCARE IMAGINI
# ============================================================

def load_images(entries, size, augment=False):
    images, labels = [], []
    for (fpath, label, _) in entries:
        img = cv2.imread(fpath)
        if img is None:
            print(f"ATENTIE: Nu s-a putut citi {fpath}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, size)
        images.append(img)
        labels.append(label)
        if augment:
            images.append(cv2.flip(img, 1))
            labels.append(label)
    return images, labels

train_images, train_labels = load_images(train_entries, IMAGE_SIZE, augment=True)
test_images,  test_labels  = load_images(test_entries,  IMAGE_SIZE, augment=False)

print(f"Train: {len(train_images)} | Test: {len(test_images)}")

In [ ]:
# ============================================================
# EXTRAGERE HOG
# ============================================================

def extract_hog(images):
    feats = []
    for img in images:
        fd = hog(
            img,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIX_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            block_norm='L2-Hys',
            transform_sqrt=True,
            feature_vector=True
        )
        feats.append(fd)
    return np.array(feats, dtype=np.float32)

X_train = extract_hog(train_images)
X_test  = extract_hog(test_images)
y_train = np.array(train_labels)
y_test  = np.array(test_labels)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

In [ ]:
# ============================================================
# ANTRENARE + EVALUARE
# ============================================================

best_model    = None
best_accuracy = 0

for C in [0.01, 0.1, 1.0, 10.0]:
    model = LinearSVC(C=C, max_iter=5000)
    model.fit(X_train, y_train)
    acc = model.score(X_train, y_train)
    print(f"C={C:.2f} -> train acc: {acc:.4f}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = deepcopy(model)

y_pred   = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Vizualizare scoruri
scores = best_model.decision_function(X_train)
plt.figure(figsize=(10, 4))
plt.plot(np.sort(scores[y_train == 1]), label='Clasa 1 (pozitiva)')
plt.plot(np.sort(scores[y_train == 0]), label='Clasa 0 (negativa)')
plt.axhline(0, color='k', linestyle='--')
plt.legend()
plt.title('Distributia scorurilor pe train')
plt.show()